## 01. 라이브러리 불러오기

In [1]:
# ==================================================
# 1. 라이브러리 불러오기
# ==================================================

import os

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image

## 02. 기본 설정

In [2]:
# ==================================================
# 2. 기본 설정
# ==================================================

# 데이터셋 위치
dataset_path = "../dataset"

# 모델 저장 위치
model_path = "models"

# 이미지 크기
image_size = 128

# 한 번에 학습할 이미지 수
batch_size = 4

# 학습 횟수
epochs = 20

# 학습률
learning_rate = 0.001


# 모델 저장 폴더 만들기
os.makedirs(
    model_path,
    exist_ok=True
)

## 03. 이미지 전처리

In [3]:
# ==================================================
# 3. 이미지 전처리
# ==================================================

transform = transforms.Compose([

    transforms.Resize(
        (image_size, image_size)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor()
])

## 04. 전체 데이터 확인

In [4]:
# ==================================================
# 4. 데이터셋 확인
# ==================================================

dataset = datasets.ImageFolder(
    dataset_path,
    transform=transform
)


print("AI가 알고 있는 물건:")
print(dataset.classes)

print()

print("전체 이미지:")
print(len(dataset))

AI가 알고 있는 물건:
['nipper', 'pen', 'wire stripper']

전체 이미지:
271


## 05. CNN 모델 정의

In [6]:
nn.Linear(128, 2)

Linear(in_features=128, out_features=2, bias=True)

In [7]:
# ==================================================
# 5. CNN 모델 정의
# ==================================================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 16 * 16,
                128
            ),

            nn.ReLU(),

            # 0 = 다른 물건
            # 1 = 목표 물건
            nn.Linear(
                128,
                2
            )
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

## 06. 부품별 이진분류 데이터 만들기

In [8]:
target_class = "nipper"

In [9]:
# ==================================================
# 6. 부품별 이진분류 데이터 만들기
# ==================================================

def make_binary_dataset(
    target_class
):

    image_paths = []

    labels = []


    # 모든 이미지 확인
    for image_path, class_index in dataset.samples:

        # 원래 클래스 이름
        original_class = dataset.classes[
            class_index
        ]


        # 목표 물건이면 1
        if original_class == target_class:

            label = 1


        # 나머지 물건이면 0
        else:

            label = 0


        image_paths.append(
            image_path
        )

        labels.append(
            label
        )


    return image_paths, labels

## 07. 이진분류 Dataset 만들기

In [10]:
# ==================================================
# 7. 이진분류 Dataset
# ==================================================

class BinaryObjectDataset(
    torch.utils.data.Dataset
):

    def __init__(
        self,
        image_paths,
        labels,
        transform=None
    ):

        self.image_paths = image_paths

        self.labels = labels

        self.transform = transform


    def __len__(self):

        return len(
            self.image_paths
        )


    def __getitem__(
        self,
        index
    ):

        # 이미지 경로
        image_path = self.image_paths[
            index
        ]


        # 정답
        label = self.labels[
            index
        ]


        # 이미지 불러오기
        image = Image.open(
            image_path
        ).convert("RGB")


        # 이미지 전처리
        if self.transform:

            image = self.transform(
                image
            )


        # 정답을 Tensor로 변환
        label = torch.tensor(
            label,
            dtype=torch.long
        )


        return image, label